# Model 2: Quality & Confidence Classifier
## Random Forest for Detecting Suspicious Measurements

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Quality Dataset

In [2]:
# Load train/val/test splits
train_df = pd.read_csv('quality_train_20260209_220137.csv')
val_df = pd.read_csv('quality_val_20260209_220137.csv')
test_df = pd.read_csv('quality_test_20260209_220137.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nLabel distribution (train):")
print(train_df['confidence_flag'].value_counts())
train_df.head()

Train: 5665 | Val: 1214 | Test: 1214

Label distribution (train):
confidence_flag
SUSPICIOUS    4034
OK            1631
Name: count, dtype: int64


,child_id,visit_date,sex,age_months,muac_mm,edema,appetite,danger_signs,label_pathway,confidence_flag,error_type,near_threshold,unit_suspect,age_suspect
0,CH001154,2023-12-16,M,18.0,114.0,0,good,0,OTP,OK,none,1,0,0
1,CH001059,2022-02-18,F,10.0,120.0,-1,good,0,TSFP,SUSPICIOUS,missing_field,0,0,0
2,CH001216,2022-05-17,M,24.0,110.0,0,good,0,OTP,OK,none,0,0,0
3,CH000441,2024-10-16,F,2.0,116.0,0,good,0,TSFP,SUSPICIOUS,age_error,1,0,1
4,CH002234,2024-01-27,F,32.0,114.0,-1,good,1,SC_ITP,SUSPICIOUS,missing_field,1,0,0


## 2. Feature Engineering

In [3]:
def prepare_features(df):
    """Encode categorical features and select training columns."""
    df = df.copy()
    
    # Encode sex
    df['sex_encoded'] = (df['sex'] == 'M').astype(int)
    
    # Encode appetite
    appetite_map = {'good': 0, 'poor': 1, 'unknown': 2}
    df['appetite_encoded'] = df['appetite'].map(appetite_map).fillna(2)
    
    # Fill missing age with median
    df['age_months'] = df['age_months'].fillna(df['age_months'].median())
    
    # Feature columns
    feature_cols = [
        'muac_mm', 'age_months', 'sex_encoded', 'edema', 
        'appetite_encoded', 'danger_signs',
        'near_threshold', 'unit_suspect', 'age_suspect'
    ]
    
    X = df[feature_cols]
    y = (df['confidence_flag'] == 'SUSPICIOUS').astype(int)  # 1=SUSPICIOUS, 0=OK
    
    return X, y, feature_cols

X_train, y_train, feature_cols = prepare_features(train_df)
X_val, y_val, _ = prepare_features(val_df)
X_test, y_test, _ = prepare_features(test_df)

print(f"Features: {feature_cols}")
print(f"\nX_train shape: {X_train.shape}")
print(f"y_train distribution: OK={sum(y_train==0)}, SUSPICIOUS={sum(y_train==1)}")

Features: ['muac_mm', 'age_months', 'sex_encoded', 'edema', 'appetite_encoded', 'danger_signs', 'near_threshold', 'unit_suspect', 'age_suspect']

X_train shape: (5665, 9)
y_train distribution: OK=1631, SUSPICIOUS=4034


## 4. Train Selected Model (Random Forest)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

# ── Candidate models ───────────────────────────────────────────────────────
# Binary labels (0=OK, 1=SUSPICIOUS) are already integers — no extra encoding needed
candidates_q = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_split=10, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=6, random_state=42,
        eval_metric='logloss', verbosity=0),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        random_state=42, n_jobs=-1),
}

comparison_results_q = {}

print("="*70)
print("MODEL COMPARISON — VALIDATION SET")
print("="*70)

for name, clf in candidates_q.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    f1  = f1_score(y_val, y_pred, average='weighted')
    comparison_results_q[name] = {'Accuracy': acc, 'F1-Score': f1}
    print(f"\n{name}:")
    print(f"  Accuracy          : {acc*100:.1f}%")
    print(f"  Weighted F1-Score : {f1*100:.1f}%")

print("\n" + "="*70)
print("Summary Table")
print("="*70)
summary_q = pd.DataFrame(comparison_results_q).T
summary_q.index.name = 'Model'
print(summary_q.applymap(lambda v: f"{v*100:.1f}%"))

## 5. Evaluate on Validation Set

## 3. Train Random Forest

## 6. Confusion Matrix

## 4. Evaluate on Validation Set

## 7. Feature Importance

## 5. Confusion Matrix

## 8. Test Set Evaluation

## 6. Feature Importance

## 9. Error Analysis

## 7. Test Set Evaluation

## 10. Save Model

## 8. Error Analysis

In [9]:
# Analyze misclassifications
test_df_copy = test_df.copy()
test_df_copy['predicted'] = ['SUSPICIOUS' if p == 1 else 'OK' for p in y_test_pred]
test_df_copy['correct'] = (y_test == y_test_pred)

# False negatives (SUSPICIOUS predicted as OK) - DANGEROUS
false_negatives = test_df_copy[(test_df_copy['confidence_flag'] == 'SUSPICIOUS') & 
                                (test_df_copy['predicted'] == 'OK')]

print(f"False Negatives (missed errors): {len(false_negatives)}")
if len(false_negatives) > 0:
    print(f"\nError types missed:")
    print(false_negatives['error_type'].value_counts())
    print(f"\nSample false negatives:")
    print(false_negatives[['muac_mm', 'age_months', 'error_type', 'unit_suspect', 'age_suspect']].head())

False Negatives (missed errors): 121

Error types missed:
error_type
noise            72
missing_field    49
Name: count, dtype: int64

Sample false negatives:
    muac_mm  age_months     error_type  unit_suspect  age_suspect
4     116.0         NaN  missing_field             0            0
25    115.0         NaN  missing_field             0            0
37    109.0        26.0          noise             0            0
39    102.0        31.0          noise             0            0
42    120.0        38.0          noise             0            0


## 9. Save Model

In [10]:
# Save model
model_path = 'model2_quality_classifier.pkl'
joblib.dump(model, model_path)
print(f"✓ Model saved: {model_path}")

# Save feature names
import json
metadata = {
    'model_type': 'RandomForestClassifier',
    'target': 'confidence_flag (OK=0, SUSPICIOUS=1)',
    'features': feature_cols,
    'train_samples': len(X_train),
    'val_accuracy': float(val_acc),
    'test_accuracy': float(test_acc),
    'n_estimators': 100,
    'max_depth': 10
}

with open('model2_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved: model2_metadata.json")

✓ Model saved: model2_quality_classifier.pkl
✓ Metadata saved: model2_metadata.json
